**Now, we will stratify the data on the basis of different indicator values, including volatility, trading volume, and market sector**

These distinct strata will allow our models to better generalize trends between similar data and avoid any confusion

First, we make necessary imports

In [1]:
import glob
import os
import json
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
files = glob.glob('data_raw/*.parquet')
stats = []

for f in files:
    df = pd.read_parquet(f)
    
    stats.append({
        'path': f,
        'vol': df['VOL_20'].median(),
        'dollar_vol': (df['Close'] * df['Volume']).mean(),
        'sector': df['Market_Category'].iloc[0]
    })
    
df_stats = pd.DataFrame(stats)

# 2 vol tiers (median split) instead of 3 — fewer, larger buckets
df_stats['vol_tier'] = pd.qcut(df_stats['vol'], 2, labels=['low_vol', 'high_vol'])

df_stats['liq_tier'] = pd.qcut(df_stats['dollar_vol'], 2, labels=['mid_liq', 'high_liq'])

MIN_BUCKET_SIZE = 30

stratified_map = {}
small_buckets = {}

for (sector, vol, liq), group in df_stats.groupby(['sector', 'vol_tier', 'liq_tier']):
    bucket_name = f"{sector}_{vol}_{liq}".replace(" ", "_")
    if len(group) >= MIN_BUCKET_SIZE:
        stratified_map[bucket_name] = group['path'].tolist()
    else:
        small_buckets[bucket_name] = (sector, vol, liq, group['path'].tolist())

# Merge small buckets into the nearest same-sector neighbor (relax liq first, then vol)
for name, (sector, vol, liq, paths) in small_buckets.items():
    # Try same sector + same vol, other liq
    other_liq = 'high_liq' if liq == 'mid_liq' else 'mid_liq'
    merge_target = f"{sector}_{vol}_{other_liq}".replace(" ", "_")
    if merge_target not in stratified_map:
        # Try same sector + other vol, same liq
        other_vol = 'high_vol' if vol == 'low_vol' else 'low_vol'
        merge_target = f"{sector}_{other_vol}_{liq}".replace(" ", "_")
    if merge_target not in stratified_map:
        # Last resort: same sector + other vol + other liq
        merge_target = f"{sector}_{other_vol}_{other_liq}".replace(" ", "_")
    
    if merge_target in stratified_map:
        print(f"Merged {name} ({len(paths)} tickers) → {merge_target}")
        stratified_map[merge_target].extend(paths)
    else:
        # No valid merge target — keep it as its own bucket with a warning
        print(f"WARNING: {name} ({len(paths)} tickers) has no merge target, keeping as-is")
        stratified_map[name] = paths

with open('stratified_metadata.json', 'w') as j:
    json.dump(stratified_map, j, indent=4)

print(f"\nCreated {len(stratified_map)} training buckets:")
for k, v in sorted(stratified_map.items()):
    print(f"  {k}: {len(v)} tickers")

Merged S_low_vol_high_liq (28 tickers) → S_low_vol_mid_liq

Created 11 training buckets:
  G_high_vol_high_liq: 186 tickers
  G_high_vol_mid_liq: 148 tickers
  G_low_vol_high_liq: 190 tickers
  G_low_vol_mid_liq: 317 tickers
  Q_high_vol_high_liq: 237 tickers
  Q_high_vol_mid_liq: 125 tickers
  Q_low_vol_high_liq: 641 tickers
  Q_low_vol_mid_liq: 199 tickers
  S_high_vol_high_liq: 156 tickers
  S_high_vol_mid_liq: 586 tickers
  S_low_vol_mid_liq: 91 tickers


**Now, we have stratified training sets for each distinct market sector (2 vol tiers × 2 liq tiers × 3 sectors = up to 12 buckets, with small buckets merged into nearest neighbors to ensure ≥30 tickers per bucket for robust cross-sectional normalization)**